In [0]:
%run "../00. Common/Config"

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.circuits"
silver_table = f"{catalog_name}.{silver_schema}.circuits"


In [0]:
circuits_df = spark.table(bronze_table)

In [0]:
display(circuits_df)

In [0]:
circuits_selected_df = circuits_df.select(
    "circuitId",
    "circuitName",
    "lat",
    "long",
    "locality",
    "country",
    "ingestion_timestamp",
    "source_file"
)

In [0]:
circuits_renamed_df = (
    circuits_selected_df
        .withColumnsRenamed({
            "circuitId": "circuit_id",
            "circuitName": "circuit_name",
            "lat": "latitude",
            "long": "longtitude"
        })
)

In [0]:
circuits_valid_df = circuits_renamed_df.filter("circuit_id IS NOT NULL")

In [0]:
circuits_distinct_df = circuits_valid_df.dropDuplicates(["circuit_id"])

In [0]:
display(circuits_distinct_df)

In [0]:
import pyspark.sql.functions as F
circuits_final_df = (
    circuits_distinct_df
        .withColumn('circuit_name',F.initcap("circuit_name"))
        .withColumn('locality',F.initcap("locality"))
)

In [0]:
(
    circuits_final_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))